# Day 66 - HNSW: finding the nearest vectors without looking at most of them

A brute-force search reads every vector. HNSW reads a few hundred and usually returns
the same answer. This notebook builds the index from scratch - the layered graph, the
greedy descent, the ef-sized candidate list, the neighbour heuristic - and measures the
three things that decide whether it was a good idea: recall, distance computations, and
what happens after you delete things.

(Building graphs in pure Python is slow; the whole notebook is a couple of minutes.)

## 1. Build one, and compare it with brute force

In [ ]:
import numpy as np
from hnsw import (HNSW, make_vectors, brute_force, brute_cost, recall_at_k,
                  ef_curve, layer_profile, without_layers, m_sweep,
                  heuristic_compare, deletion_decay, scaling, ef_below_k,
                  forgot_to_normalise, circular_ground_truth, check)

world = make_vectors(docs=3_000, queries=120)
pts, qs = world['points'], world['queries']
gold = brute_force(pts, qs, 10)
index = HNSW(world['cfg']['dims'], M=12, ef_construction=60).build(pts)
curve = ef_curve(index, pts, qs, gold)
bf = brute_cost(pts, qs)
at50 = [r for r in curve if r['ef'] == 50][0]
print(f"{len(pts):,} vectors, {world['cfg']['dims']} dimensions, build {index.build_seconds:.1f} s\n")
print(f"brute force : {bf['dist_per_query']:7,.0f} distances   recall {bf['recall']:.3f}")
print(f"HNSW ef=50  : {at50['dist_per_query']:7,.0f} distances   recall {at50['recall']:.3f}"
      f"   ({at50['fraction_seen']:.0%} of the corpus)")

3,000 vectors, 48 dimensions, build 12.0 s

brute force :   3,000 distances   recall 1.000
HNSW ef=50  :     908 distances   recall 0.947   (30% of the corpus)


## 2. The layers

Each level up holds about 1/M of the one below - a skip list with a graph instead of a
linked list.

In [ ]:
for r in layer_profile(index):
    print(f"layer {r['layer']}  {r['nodes']:6,} nodes   {r['share']:6.3f} of the corpus"
          f"   expected {r['expected']:8,.1f}")

layer 0   3,000 nodes    1.000 of the corpus   expected  3,000.0
layer 1     260 nodes    0.087 of the corpus   expected    250.0
layer 2      21 nodes    0.007 of the corpus   expected     20.8
layer 3       5 nodes    0.002 of the corpus   expected      1.7


## 3. ef, the only query-time knob

In [ ]:
for r in curve:
    print(f"ef {r['ef']:4}   {r['dist_per_query']:7,.0f} distances   "
          f"{r['fraction_seen']:6.1%} of corpus   recall {r['recall']:.3f}")

ef   10       319 distances    10.6% of corpus   recall 0.629
ef   25       567 distances    18.9% of corpus   recall 0.849
ef   50       908 distances    30.2% of corpus   recall 0.947
ef  100     1,432 distances    47.7% of corpus   recall 0.996
ef  200     2,071 distances    69.0% of corpus   recall 1.000


## 4. The hierarchy is not where the speed comes from

Searching the bottom layer only - even from a random entry point - costs the same and
returns the same answers. This reproduces a published result (ECIR 2026): the hub nodes
in the bottom graph already act as the express lanes.

In [ ]:
for r in without_layers(index, qs, gold):
    print(f"{r['name']:40} {r['dist_per_query']:7,.0f} distances   recall {r['recall']:.3f}")

with the express lanes                       908 distances   recall 0.947
bottom layer only, same entry point          886 distances   recall 0.947
bottom layer only, random entry point        886 distances   recall 0.947


## 5. M: neighbours, memory and build time

In [ ]:
small = make_vectors(seed=1, docs=1_000, queries=80)
gold_s = brute_force(small['points'], small['queries'], 10)
for r in m_sweep(small['points'], small['queries'], gold_s, Ms=(6, 12, 20)):
    print(f"M {r['M']:3}  {r['edges_per_node']:5.1f} edges/node  "
          f"{r['link_bytes']:5.0f} link bytes ({r['overhead']:.2f}x the vector)  "
          f"build {r['build_seconds']:5.1f} s  recall {r['recall']:.3f}")

M   6   13.1 edges/node     52 link bytes (0.27x the vector)  build   1.3 s  recall 0.936
M  12   24.9 edges/node    100 link bytes (0.52x the vector)  build   3.5 s  recall 0.996
M  20   40.9 edges/node    164 link bytes (0.85x the vector)  build   8.4 s  recall 0.999


## 6. Which neighbours to keep

Plain nearest-M fills the list with one cluster; the heuristic keeps the edges that
leave it.

In [ ]:
clustered = make_vectors(seed=2, docs=1_500, queries=80, clusters=12)
gold_c = brute_force(clustered['points'], clustered['queries'], 10)
for r in heuristic_compare(clustered['points'], clustered['queries'], gold_c,
                           clustered['members'], ef=25, M=6):
    print(f"{r['name']:22} recall {r['recall']:.3f}   "
          f"edges inside the cluster {r['same_cluster_edges']:.0%}   "
          f"build {r['build_seconds']:.1f} s")

plain nearest-M        recall 0.754   edges inside the cluster 45%   build 0.9 s
heuristic (diverse)    recall 0.799   edges inside the cluster 39%   build 2.2 s


## 7. Deletions are tombstones

In [ ]:
rows = deletion_decay(make_vectors(seed=3, docs=1_500, queries=80)['points'],
                      qs[:80], fractions=(0.1, 0.3))
for r in rows:
    if r.get('rebuilt'):
        print(f"rebuilt after 30%     recall {r['recall']:.3f}   "
              f"{r['dist_per_query']:5,.0f} distances   rebuild {r['rebuild_seconds']:.1f} s")
    else:
        print(f"{r['deleted']:.0%} deleted           recall {r['recall']:.3f}   "
              f"{r['dist_per_query']:5,.0f} distances   "
              f"{r['wasted_share']:.0%} of them on deleted points")

10% deleted           recall 0.985     757 distances   10% of them on deleted points
30% deleted           recall 0.974     757 distances   30% of them on deleted points
rebuilt after 30%     recall 0.993     666 distances   rebuild 4.1 s


## 8. Growing the corpus

In [ ]:
rows = scaling(sizes=(500, 1_000, 2_000))
rows.append({'docs': len(pts), 'recall': at50['recall'],
             'dist_per_query': at50['dist_per_query'],
             'fraction_seen': at50['fraction_seen'], 'build_seconds': index.build_seconds})
for r in rows:
    print(f"{r['docs']:6,} vectors   {r['dist_per_query']:6,.0f} distances   "
          f"{r['fraction_seen']:6.1%} of corpus   recall {r['recall']:.3f}")

   500 vectors      446 distances    89.1% of corpus   recall 0.998
 1,000 vectors      644 distances    64.4% of corpus   recall 0.993
 2,000 vectors      824 distances    41.2% of corpus   recall 0.972
 3,000 vectors      908 distances    30.2% of corpus   recall 0.947


## 9. The quiet failures

In [ ]:
print(ef_below_k(index, qs, gold, k=10, ef=5))
print(circular_ground_truth(index, qs, gold))
print(forgot_to_normalise(world))

{'ef': 5, 'k': 10, 'clamped_recall': 0.6291666666666667, 'unclamped_recall': 0.4033333333333333, 'queries_returning_fewer_than_k': 1.0}
{'against_brute_force': 0.9466666666666667, 'against_itself': 1.0}
{'recall_against_true_cosine': 0.25, 'share_scaled': 0.25133333333333335, 'share_of_hits_scaled': 1.0}


## 10. Assertions

In [ ]:
check()
print('all assertions passed')

all assertions passed
